# Corrective Fine-Tune Run (checkpoint-8000)

This notebook runs a targeted corrective continuation to improve pixel integrity, prompt following, and single-subject behavior.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from pathlib import Path

REPO = Path('/content/Text-to-Image-Pixel-Art')
if not REPO.exists():
    !git clone https://github.com/arifdag/Text-to-Image-Pixel-Art.git {REPO}

%cd /content/Text-to-Image-Pixel-Art
!git checkout main
!git pull origin main


In [ ]:
%cd /content/Text-to-Image-Pixel-Art
!python -m pip install -q -r requirements.txt -r requirements-dev.txt
!python -m pip install -q -e .


In [ ]:
%cd /content/Text-to-Image-Pixel-Art
!rm -rf /content/diffusers
!git clone -q https://github.com/huggingface/diffusers.git /content/diffusers
!python -m pip uninstall -y -q diffusers
!python -m pip install -q -e /content/diffusers


In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path('/content/Text-to-Image-Pixel-Art')
commands = [
    [sys.executable, '-m', 'pixelart.data_ingest', '--config', 'configs/data_sources_corrective.yaml', '--output-dir', 'data/raw_corrective', '--index-out', 'data/raw_corrective/index.jsonl', '--registry-path', 'DATA_SOURCES.md', '--skip-failed-sources'],
    [sys.executable, '-m', 'pixelart.data_clean', '--input-dir', 'data/raw_corrective', '--index-in', 'data/raw_corrective/index.jsonl', '--output-dir', 'data/clean_corrective/images', '--index-out', 'data/clean_corrective/index.jsonl', '--rejects-out', 'data/clean_corrective/rejects.jsonl', '--resolution', '1024', '--resize-mode', 'pad', '--max-upscale-factor', '8', '--phash-threshold', '6', '--edge-softness-threshold', '0.62', '--max-source-dimension', '384', '--max-aspect-ratio', '1.8', '--reject-name-patterns', 'sheet,spritesheet,sprite_sheet,atlas,tilemap,tilesheet'],
    [sys.executable, '-m', 'pixelart.caption', '--index-in', 'data/clean_corrective/index.jsonl', '--output-dir', 'data/train_corrective', '--metadata-out', 'data/train_corrective/metadata.jsonl', '--prefer-source-prompt', '--style-tags', 'pixel art,single subject,limited palette,crisp edges,retro game style'],
]

for cmd in commands:
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=repo)

raw_index = repo / 'data/raw_corrective/index.jsonl'
raw_count = 0
if raw_index.exists():
    raw_count = sum(1 for line in raw_index.read_text(encoding='utf-8').splitlines() if line.strip())
if raw_count < 400:
    raise RuntimeError(f'Only {raw_count} raw images ingested. Update data_sources_corrective URLs before training.')
print('Raw ingest count:', raw_count)


In [ ]:
import json
from collections import Counter
from pathlib import Path

kept = Path('data/clean_corrective/index.jsonl').read_text(encoding='utf-8').strip().splitlines()
reject_lines = Path('data/clean_corrective/rejects.jsonl').read_text(encoding='utf-8').strip().splitlines()
reject_rows = [json.loads(x) for x in reject_lines if x.strip()]
print('Kept images:', len([x for x in kept if x.strip()]))
print('Rejected images:', len(reject_rows))
print('Top reject reasons:', Counter(r.get('reason', 'unknown') for r in reject_rows).most_common(10))


In [ ]:
%cd /content/Text-to-Image-Pixel-Art
!python -m pixelart.train --config configs/train_sdxl_lora_corrective.yaml


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import yaml

repo = Path('/content/Text-to-Image-Pixel-Art')
os.chdir(repo)

run_root = Path('/content/drive/MyDrive/pixelart-lora-output-corrective-ckpt8000')
checkpoints = sorted([p for p in run_root.glob('checkpoint-*') if p.is_dir()], key=lambda p: int(p.name.split('-')[-1]))
if not checkpoints:
    raise RuntimeError(f'No checkpoints found in {run_root}')
latest = checkpoints[-1]
print('Using checkpoint for eval:', latest)

cfg = yaml.safe_load(Path('configs/eval_corrective.yaml').read_text(encoding='utf-8'))
cfg['lora_path'] = str(run_root)
cfg['checkpoint_subdir'] = latest.name
cfg['output_dir'] = f"artifacts/eval_corrective_{latest.name}"
cfg['eval_mode'] = 'memory_safe'
cfg_path = Path('configs') / f"eval_corrective_{latest.name}.yaml"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

result = subprocess.run([sys.executable, '-m', 'pixelart.eval', '--config', str(cfg_path)], capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'eval failed with exit code {result.returncode}')


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

eval_dirs = sorted(Path('artifacts').glob('eval_corrective_checkpoint-*'))
if not eval_dirs:
    eval_dirs = sorted(Path('artifacts').glob('eval_corrective_*'))
if not eval_dirs:
    raise RuntimeError('No corrective eval directory found in artifacts/')
eval_dir = eval_dirs[-1]
grid_files = sorted((eval_dir / 'grids').glob('*.png'))
print('Eval directory:', eval_dir)
print('Grid count:', len(grid_files))

preview = grid_files[:6]
if not preview:
    print('No grid files found.')
else:
    fig, axes = plt.subplots(len(preview), 1, figsize=(14, 3 * len(preview)))
    if len(preview) == 1:
        axes = [axes]
    for ax, path in zip(axes, preview):
        ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(path.name)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
import json
from collections import Counter
from pathlib import Path

eval_dirs = sorted(Path('artifacts').glob('eval_corrective_*'))
if not eval_dirs:
    raise RuntimeError('No corrective eval outputs to summarize.')
report_path = eval_dirs[-1] / 'report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
rows = report.get('results', [])
print('Report:', report_path)
print('Results:', len(rows))
print('Categories:', Counter(r.get('category', 'unknown') for r in rows))
print('Checkpoint used:', report.get('checkpoint_used'))
print('Resolved lora path:', report.get('config', {}).get('resolved_lora_path'))
